# Published Benchmarks

Two games from the literature with partial information and an exact solution of the discretised game. In both, one
player's information is not contained in the other's.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, gradient_play, linear_equilibrium, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import BrokerTrader, MajorMinor

## 1. Broker and informed trader (Wu and Jaimungal, 2024)

An informed trader observes a mean-reverting signal $\alpha$ and trades $\eta$ with a broker, who observes the price
only and trades $\nu$ on the exchange; here without transient impact.

- Price and signal: $dz_t = \alpha_t\,dt + \sigma\,dW_t$, $d\alpha_t = -\kappa\,\alpha_t\,dt + \sigma_\alpha\,dB_t$.
- Inventories: $dQ^I_t = \eta_t\,dt$, $dQ^B_t = (\nu_t - \eta_t)\,dt$.
- The trader maximizes $\mathbb{E}\big[\int_0^T (Q^I_t\alpha_t - b\,\eta_t^2 - r_I (Q^I_t)^2)\,dt - \psi (Q^I_T)^2\big]$ and the broker the same with $(Q^B, \nu, a, r_B, \phi)$.

The exact solution is the trader's discrete LQR and the broker's Kalman filter of $(\alpha, Q^I)$ followed by a discrete LQR.

In [ ]:
game = BrokerTrader()
solver = Solver(game, damping=0.7, steps=750, batch=2048, n_paths=16384, control_variate=True)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(12, average_from=6, reference=exact, log=lambda message: None)
print(f'relative L2 error: trader {relative_l2(learned[0], exact[0]):.3f}, broker {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['informed trader', 'broker'])

## 2. Major and minor player (Firoozi and Caines, 2020)

A major player and a minor player each observe their own state through noise; the minor player also observes the
major player's state through noise, and its dynamics depend on that state.

- States: $dx^0_t = (A_0 x^0_t + B_0 u^0_t)\,dt + D_0\,dw^0_t$, $dx^1_t = (A_1 x^1_t + G x^0_t + B_1 u^1_t)\,dt + D_1\,dw^1_t$.
- Observations: $dy^0_t = L_0 x^0_t\,dt + s_0\,dv^0_t$, $dy^1_t = (L_1 x^1_t, L_2 x^0_t)\,dt + s_1\,dv^1_t$.
- Costs: $\mathbb{E}\int_0^T e^{-\rho t}[Q_0(x^0_t - \eta_0)^2 + R_0 (u^0_t)^2]\,dt$ and $\mathbb{E}\int_0^T e^{-\rho t}[Q_1(x^1_t - H x^0_t - \eta_1)^2 + R_1 (u^1_t)^2]\,dt$.

The minor player must estimate the major player's own estimate of $x^0$. The exact solution is the discrete Kalman
filter and discounted LQR of each player.

In [ ]:
game = MajorMinor()
solver = Solver(game, damping=0.7, steps=750, batch=2048, n_paths=16384, control_variate=True)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(12, average_from=6, reference=exact, log=lambda message: None)
print(f'relative L2 error: major {relative_l2(learned[0], exact[0]):.3f}, minor {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['major player', 'minor player'])